In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

#import personnal tools
import sys
sys.path.append('../tools/')
from info import *
from imports import *
from tools_generic import *
from events import *

# Load files

In [ ]:
site_list=["d17","d47","d85","dmc"]
file_start_date = '20241201'
file_end_date = '20260630'

golden_start_date="2025-02-01"
golden_end_date="2025-02-28"

In [ ]:
data = {}
daily_data = {}

In [ ]:
# data, daily_data = create_dict_data_datadaily(sites, sensors, start_date, end_date)
data = create_data(site_list, sensors, file_start_date, file_end_date)

In [ ]:
# open wind_beginning files
var_list=["wspd1","wspd2", "wdir"]#,"wspd3"]
for site in site_list:
    file_path=f'../../data/WIND_BEGINNING_{site}_20241201_20250313.netcdf'
    
    ds = xr.open_dataset(file_path, engine='netcdf4')
   
    for var in var_list:
        # Check if the variable exists in both datasets
        if var in data["WIND"][site] and var in ds:
            wind_var = data["WIND"][site][var]
            ds_var = ds[var]

            # Align the datasets along the time dimension
            wind_var_aligned, ds_var_aligned = xr.align(wind_var, ds_var, join='outer')

            # Merge the datasets: prioritize non-NaN values from wind_var, fall back to ds_var
            merged_var = wind_var_aligned.where(~np.isnan(wind_var_aligned), ds_var_aligned)

            # Update the data["WIND"][site][var] with the merged result
            data["WIND"][site][var] = merged_var
        else:
            print(f"Variable {var} not found in both datasets for site {site}")

    ds.close()  # Close the dataset to free resources

In [ ]:
# Extract period of interest (golden month)
data = filter_datasets_golden(
    data, start_date=golden_start_date, end_date=golden_end_date
 )

In [ ]:
# create daily
daily_data = create_daily_data(data)

In [ ]:
data["SURF"]["d17"]

## Filter outliers

In [ ]:
varlist=['snowflux']
data = filter_data_by_max_values(
    data, 
    varlist
    )

# Stats

In [ ]:
variable = 'FluxMean1'
compute_variable_stats(data, variable)

In [ ]:
var="FluxMean1"
plot_binned_distribution(data, var, variable_to_sensor, bin_number=30, min_value=0, max_value=200)

In [ ]:
var="snowflux"
plot_binned_distribution(data, var, variable_to_sensor, bin_number=30, min_value=0, max_value=200)

# Basic plotting

## Time plots

In [ ]:
variables=["FluxMean1", "FluxMean2", 'snowflux', 'Hagl', 'wspd1','wspd2', ]
variables = ['wspd1','wspd2']
variables = ["FluxMean1",'FluxMean2']
variables= ["snowflux", 'Hagl'] 
variables = ['T1','RH1']
variables=["wdir",'wspd1']

plot_per_var_multiple_sites(
    # sensor_datasets=daily_data,
    sensor_datasets=data,
    variables = variables,
    # sites=sites,
    sites=["d17","d47","d85"],
    # figsize=(15, 3),
    ymin=[0,0], ymax=[360,26]
    # ymin=[-80, -80, -80],  # Custom ymin for each variable
    # ymax=[350, 4], # Custom ymax for each variable
    # colors=["blue", "red"],  # Valid color strings (e.g., hex or named colors)
)

In [ ]:
vars=["FluxMean1", "FluxMean2", "snowflux"]
# vars=['wspd1','wspd2','wspd3']
# vars=

plot_per_site_multiple_vars(
    data,
    vars,
    sites=["d17","d47"],
    figsize=(15, 5),
    ymax=300
)

## Scatters

In [ ]:
var1='wdir'
var2='wspd1'
site1="d17"
site2="d17"
plot_bivariate_scatter(
    data,
    var1=var1,
    var2=var2,
    site1=site1,
    site2=site2,
    show_corr= False,
    show_fit=False,
    min_val=[0,0],
    max_val=[360,26],
    figsize = (7, 6),
    # show_oneone=True
)

In [ ]:
var1='FluxMean2'
var2='snowflux'
var3='wspd1'
site1="d47"
site2="d47"
site3="d47"
plot_trivariate_scatter(
    data,
    var1=var1,
    var2=var2,
    var3=var3,
    site1=site1,
    site2=site2,
    site3=site3,
    min3=10,
    max3=20,
    # show_corr= False,
    # show_fit=False,
    max_val=[300, 300],
    figsize = (7, 6),
    show_oneone=True
)

# Events

In [ ]:
# 1. Instantiate detector with your threshold & parameters
detector = EventDetector(
    threshold=1.0,
    min_timesteps=12, #NB: depends on resampling of timesteps
    buffer_timesteps=0,
    variable_to_sensor=variable_to_sensor,
)

# 2. Run detection across all sites
sampled_dict = create_resampled_data(data, '30min')
# sampled_dict = data

collection = detector.detect_events(sampled_dict,
                                    variable="FluxMean2",
                                    additional_variables=["FluxMean1", "snowflux", "wspd1",'wspd2','wdir','Hagl','T1','RH1'])
collection_d17 = collection.get_site('d17')
collection_d47 = collection.get_site('d47')
non_collection = detector.detect_non_events(sampled_dict, variable="FluxMean2",
                                    additional_variables=["FluxMean1", "snowflux", "wspd1",'wspd2','wdir','Hagl','T1','RH1'])
non_collection_d17 = non_collection.get_site('d17')
non_collection_d47 = non_collection.get_site('d47')

In [ ]:
collection_d17.to_catalog(variables=["FluxMean2"])#.head()

In [ ]:
non_collection_d17.to_catalog(variables=["FluxMean2"])#.head()

In [ ]:
collection_d47.to_catalog(variables=["FluxMean2"])

In [ ]:
# plot_single_event(collection[2], 
#                     ['FluxMean1','FluxMean2','snowflux','wspd1'])

In [ ]:
print_duration_stats(collection)
print_duration_stats(collection_d17)
print_duration_stats(collection_d47)

In [ ]:
print_integrated_flux_stats(collection)
print_integrated_flux_stats(collection_d17)
print_integrated_flux_stats(collection_d47)

In [ ]:
plot_diurnal_start_distribution(collection_d17)

### Time series

In [ ]:
plot_event_collection_traces(
    collection=collection_d17,
    variable="wspd1",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.8,
    show_mean = False
)

In [ ]:
plot_event_collection_traces(
    collection=non_collection_d17,
    variable="FluxMean2",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.7,
)

In [ ]:
plot_event_collection_traces(
    collection=collection_d47,
    variable="wspd1",
    align_to="start_time",
    time_unit="h",
    cmap_name="viridis",
    alpha=1,
    linewidth=0.7,
    show_mean = False

)

In [ ]:
varlist = ['FluxMean2','FluxMean1','snowflux']
varlist = ['FluxMean2', 'wspd1','wdir']
varlist = ['Hagl','T1','RH1']
plot_events_vs_nonevents_chronological(
    collection_d17,
    non_collection_d17,
    variables=varlist,
    figsize=(15,15)
)
plot_events_vs_nonevents_chronological(
    collection_d47,
    non_collection_d47,
    variables=varlist,
    figsize=(15,15)
)

In [ ]:
varlist = ['FluxMean2','FluxMean1','snowflux','wspd1']
varlist = ['FluxMean2','Hagl','T1','RH1']
# plot_events_vs_nonevents_chronological(
#     collection_d47,
#     non_collection_d47,
#     variables=varlist
# )

In [ ]:
var1='wspd1'
var2='FluxMean2'
plot_collection_bivariate_scatter(
    collection = collection,
    var1 = var1,     var2 = var2,
    show_corr=False,     show_fit = False
)

plot_collection_bivariate_scatter(
    collection = non_collection,
    var1 = var1,     var2 = var2,
    show_corr=False,     show_fit = False
)

### Composites

In [ ]:
composite = compute_event_composite(
    events=collection,
    variables=['FluxMean1','FluxMean2','snowflux'],
    align_to="start_time",
    time_unit="h",
)
composite_d17 = compute_event_composite(
    events=collection_d17,
    variables=['FluxMean1','FluxMean2','snowflux'],
    align_to="start_time",
    time_unit="h",
)
composite_d47 = compute_event_composite(
    events=collection_d47,
    variables=['FluxMean1','FluxMean2','snowflux'],
    align_to="start_time",
    time_unit="h",
)


In [ ]:
plot_event_composite(
    composite_ds=composite,
    variables=['FluxMean1','FluxMean2','snowflux', 'wspd1'],
    use_quantiles=True,
)

In [ ]:
plot_event_composite(
    composite_ds=composite_d17,
    variables=['FluxMean1','FluxMean2','snowflux','wspd1'],
    use_quantiles=True,
)

In [ ]:
plot_event_composite(
    composite_ds=composite_d47,
    variables=['FluxMean1','FluxMean2','snowflux','wspd1'],
    use_quantiles=True,
)

# MRR

In [ ]:
mrr_site="d17"
mrr_filename=f'../../data/MRR_aggregated/mrr_{mrr_site}_20250201_20250228.nc'
mrr = xr.open_dataset(mrr_filename)
mrr

In [ ]:
## If need to reextract data 

# zea = mrr['Zea']
# zea_linear = 10 ** (zea / 10)
# zea_lin_10mn = zea_linear.resample(time='10min').mean(dim='time')
# zea_dbz_10mn = 10 * np.log10(zea_lin_10mn)
# zea_dbz_10mn.to_netcdf("../../data/MRR_aggregated/zea_averaged10mn_mrr_d17_20250201_20250228.nc")

## If already resampled and created

zea_dbz_10mn = xr.open_dataset(f'../../data/MRR_aggregated/zea_averaged10mn_mrr_{mrr_site}_20250201_20250228.nc')
zea_dbz_10mn

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
format_time_plot(zea_dbz_10mn['Zea'].mean(dim='range'),ax)